In [10]:
import torch
from at2v.tokenizer import TagBPETokenizer
from at2v.anitag2vec import ModelConfig, AniTag2VecRunner, AniTag2Vec
import torch.nn.functional as F
from at2v.dloader import MergeSet

In [11]:
TOKENIZER_PATH = "../checkpoints/token_dataset_b0d065e705028cb3_vocab_size_5000_freq_3.json"
CONFIG_PATH = "../checkpoints/config_63fc21b89723d1ce_b0d065e705028cb3.json"
MODEL_PATH = "../checkpoints/anitag2vec_63fc21b89723d1ce_b0d065e705028cb3_i128_e14_s157043_b256_p1871744.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = ModelConfig.load_from_file(CONFIG_PATH)
print(cfg)
tagtok = TagBPETokenizer(vocab_size=cfg.HYPERP_TAGTOK_VOCAB_SIZE, min_frequency=cfg.HYPERP_TAGTOK_MIN_FREQ)
tagtok.load(TOKENIZER_PATH)

anitag2vec = AniTag2Vec(
    vocab_size=tagtok.vocab_size,
    max_len_cut=cfg.HYPERP_TAGTOK_MAX_TOKEN_CLAMP,
    d_model=cfg.HYPERP_TRANSFORMER_D_MODEL,
    n_heads=cfg.HYPERP_TRANSFORMER_N_HEADS,
    n_layers=cfg.HYPERP_TRANSFORMER_N_LAYERS,
    output_emb=cfg.HYPERP_OUTPUT_EMB,
)
anitag2vec.to(device)
anitag2vec.load_state_dict(torch.load(MODEL_PATH))
anitag2vec.eval()
runner = AniTag2VecRunner(tagtok, anitag2vec)

ModelConfig(HYPERP_TAGTOK_MAX_TOKEN_CLAMP=128, HYPERP_TAGTOK_VOCAB_SIZE=5000, HYPERP_TAGTOK_MIN_FREQ=3, HYPERP_TRANSFORMER_D_MODEL=128, HYPERP_TRANSFORMER_N_HEADS=8, HYPERP_TRANSFORMER_N_LAYERS=2, HYPERP_OUTPUT_EMB=128)


In [12]:
data = MergeSet.from_file(f"../data/output/merged_tags_v2.json")
runner = AniTag2VecRunner(tagtok, anitag2vec)

In [13]:
def compare(a: str, b: str):
    ax = runner.run_inference_human([a])
    bx = runner.run_inference_human([b])
    howmuch = ((F.normalize(ax) @ F.normalize(bx).T).item())
    print(f"{howmuch:.2f}: '{a}' vs '{b}'")

In [14]:
print("Permutation invariance:")
compare("#1girl #1boy", "#1boy #1girl")
compare("#nijigasaki #rina_tennoji", "#rina_tennoji #nijigasaki")

print("\nOppositeness and likeliness:")
compare("#nikke", "#blue_archive")
compare("#hayase_yuuka", "#blue_archive")
compare("#♀", "#girl")
compare("#♂", "#boy")
compare("#girl", "#boy")
compare("#♂", "#♀")
compare("#♂", "#unrelated")

print("\nSubsetness behavior on known and new combinations:")
compare("#A #B #C", "#C #B #A")
compare("#A #B #C", "#B #C #A")
compare("#1girl", "#1boy #1girl")
compare("#1boy", "#1boy #1girl")
compare("#d #a", "#a #b #c #d #e")
compare("#a #e", "#a #b #c #d #e")
compare("#a #e #c", "#a #b #c #d #e")

print("\nVery unlikely combinations:")
compare("#foo #bar", "#bar #foo")
compare("#foo #bar", " #foo")

Permutation invariance:
1.00: '#1girl #1boy' vs '#1boy #1girl'
1.00: '#nijigasaki #rina_tennoji' vs '#rina_tennoji #nijigasaki'

Oppositeness and likeliness:
0.11: '#nikke' vs '#blue_archive'
0.51: '#hayase_yuuka' vs '#blue_archive'
0.47: '#♀' vs '#girl'
0.46: '#♂' vs '#boy'
0.50: '#girl' vs '#boy'
1.00: '#♂' vs '#♀'
0.18: '#♂' vs '#unrelated'

Subsetness behavior on known and new combinations:
1.00: '#A #B #C' vs '#C #B #A'
1.00: '#A #B #C' vs '#B #C #A'
0.81: '#1girl' vs '#1boy #1girl'
0.87: '#1boy' vs '#1boy #1girl'
0.54: '#d #a' vs '#a #b #c #d #e'
0.62: '#a #e' vs '#a #b #c #d #e'
0.75: '#a #e #c' vs '#a #b #c #d #e'

Very unlikely combinations:
1.00: '#foo #bar' vs '#bar #foo'
0.83: '#foo #bar' vs ' #foo'


In [43]:
from tqdm import tqdm
query = ["boktchi wa benkyou dekinai", "mafuyu"] # say we got some typos
batch = 1000 # oneshot 1000 embeddings is relatively cheap (1GB GPU memory)
best = []
for start in tqdm(range(0, len(data.real_examples), batch), desc="Processing"):
    end = start + batch
    ranked = runner.rank_cosim(
        query,
        data.real_examples[start:end],
        best=True
    )
    if len(ranked) > 0:
        best.append(ranked[0])
best.sort(key=lambda x: -x[0])

Processing: 100%|██████████| 13/13 [00:06<00:00,  2.04it/s]


In [44]:
print("Closest to:", ", ".join(query))
for index, (cosim, tags) in enumerate(best[:3]):
    print(f"# {index + 1}, {cosim.item():.2}: {', '.join(sorted(tags))}")

Closest to: boktchi wa benkyou dekinai, mafuyu
# 1, 0.74: arai kazuki, asumi kominami, bokutachi wa benkyou ga dekinai, doujinshi, japanese, mafuyu kirisu, maruarai
# 2, 0.7: arai kazuki, bokutachi wa benkyou ga dekinai, doujinshi, japanese, mafuyu kirisu, maruarai, nariyuki yuiga
# 3, 0.55: arai kazuki, bokutachi wa benkyou ga dekinai, doujinshi, english, glasses, maruarai, nariyuki yuiga, rizu ogata, sole male, translated
